# Autoresearch-RL Experiment Analysis

Analysis of autonomous RL post-training results from `results.tsv`.

In [2]:
import csv
import matplotlib.pyplot as plt

# Load the TSV manually (no pandas needed)
rows = []
with open("results.tsv") as f:
    reader = csv.DictReader(f, delimiter="\t")
    for row in reader:
        rows.append(row)

# Parse fields
for r in rows:
    try:
        r["eval_score"] = float(r["eval_score"])
    except (ValueError, TypeError):
        r["eval_score"] = None
    try:
        r["memory_gb"] = float(r["memory_gb"])
    except (ValueError, TypeError):
        r["memory_gb"] = None
    r["status"] = r["status"].strip().lower()

print(f"Total experiments: {len(rows)}")

# Count outcomes
from collections import Counter
counts = Counter(r["status"] for r in rows)
print("Experiment outcomes:")
for status, count in counts.most_common():
    print(f"  {status}: {count}")

n_keep = counts.get("keep", 0)
n_discard = counts.get("discard", 0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# Show all KEPT experiments (the improvements that stuck)
kept = [r for r in rows if r["status"] == "keep"]
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in enumerate(kept):
    score = row["eval_score"]
    desc = row["description"]
    mem = row["memory_gb"]
    print(f"  #{i:3d}  score={score:.3f}  mem={mem:.1f}GB  {desc}")

## Eval Score Over Time

Track how the best (kept) eval_score evolves as experiments progress. The running maximum shows the "frontier" — the best result achieved so far.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 8))

# Filter out crashes for plotting
valid = [r for r in rows if r["status"] != "crash"]

# Separate by status with their valid-index
disc_x, disc_y = [], []
keep_x, keep_y, keep_desc = [], [], []

for i, r in enumerate(valid):
    if r["status"] == "discard":
        disc_x.append(i)
        disc_y.append(r["eval_score"])
    elif r["status"] == "keep":
        keep_x.append(i)
        keep_y.append(r["eval_score"])
        keep_desc.append(str(r["description"]).strip())

# Discarded as faint background dots
ax.scatter(disc_x, disc_y,
           c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")

# Kept experiments as prominent green dots
ax.scatter(keep_x, keep_y,
           c="#2ecc71", s=50, zorder=4, label="Kept", edgecolors="black", linewidths=0.5)

# Running maximum step line (higher is better for RL)
running_max_y = []
cur_max = float("-inf")
for y in keep_y:
    cur_max = max(cur_max, y)
    running_max_y.append(cur_max)
ax.step(keep_x, running_max_y, where="post", color="#27ae60",
        linewidth=2, alpha=0.7, zorder=3, label="Running best")

# Label each kept experiment with its description
for x, y, desc in zip(keep_x, keep_y, keep_desc):
    if len(desc) > 45:
        desc = desc[:42] + "..."
    ax.annotate(desc, (x, y),
                textcoords="offset points",
                xytext=(6, 6), fontsize=8.0,
                color="#1a7a3a", alpha=0.9,
                rotation=30, ha="left", va="bottom")

best = max(keep_y)
baseline = keep_y[0]
n_total = len(rows)
n_kept = len(keep_x)
ax.set_xlabel("Experiment #", fontsize=12)
ax.set_ylabel("Eval Score (higher is better)", fontsize=12)
ax.set_title(f"Autoresearch-RL Progress: {n_total} Experiments, {n_kept} Kept Improvements", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# Y-axis: from just below baseline to just above best
margin = (best - baseline) * 0.15 if best != baseline else 0.02
ax.set_ylim(baseline - margin, best + margin)

plt.tight_layout()
plt.savefig("progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

## Summary Statistics

In [ ]:
# Summary stats
kept = [r for r in rows if r["status"] == "keep"]
baseline_score = kept[0]["eval_score"]
best_score = max(r["eval_score"] for r in kept)
best_row = max(kept, key=lambda r: r["eval_score"])

print(f"Baseline eval_score:  {baseline_score:.3f}")
print(f"Best eval_score:      {best_score:.3f}")
print(f"Total improvement:    +{best_score - baseline_score:.3f} ({(best_score - baseline_score) / baseline_score * 100:.1f}%)")
print(f"Best experiment:      {best_row['description']}")
print()

# Cumulative effort per improvement
print("Cumulative effort per improvement:")
for i, row in enumerate(kept):
    desc = str(row["description"]).strip()
    print(f"  #{i:3d}: score={row['eval_score']:.3f}  {desc}")

## Top Hits (Kept Experiments by Improvement)

In [ ]:
# Each kept experiment's delta vs the previous kept experiment
kept = [r for r in rows if r["status"] == "keep"]
hits = []
for i in range(1, len(kept)):
    delta = kept[i]["eval_score"] - kept[i - 1]["eval_score"]
    hits.append((delta, kept[i]["eval_score"], kept[i]["description"]))

# Sort by delta improvement (biggest first)
hits.sort(key=lambda x: x[0], reverse=True)

print(f"{'Rank':>4}  {'Delta':>8}  {'Score':>8}  Description")
print("-" * 80)
total_delta = 0
for rank, (delta, score, desc) in enumerate(hits, 1):
    total_delta += delta
    print(f"{rank:4d}  {delta:+.3f}    {score:.3f}    {desc}")

print(f"\n{'':>4}  {total_delta:+.3f}    {'':>8}  TOTAL improvement over baseline")